In [1]:
#Get the data
!pip install -U -q PyDrive2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
joserfc 1.7.1 requires cryptography>=45.0.1, but you have cryptography 43.0.3 which is incompatible.


In [2]:
#File ID
zip_id = "18MKzvv2LxBVVcrJEitN-CMLLsYzurXg6"
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
import zipfile, os

In [3]:
# Authenticate and create the PyDrive client.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [4]:
# Download the zip file
myzip = drive.CreateFile({'id': zip_id})
myzip.GetContentFile('Rock-Paper-Scissors.zip')

In [5]:
#unziping the zip-file to tmp
!unzip -qq -o Rock-Paper-Scissors.zip -d /tmp

In [6]:
#Defining the directories
base_dir = '/tmp/rock-paper-scissors'

In [7]:
#Defining the train and test directories
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

In [8]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [9]:
# All images will be rescaled.
train_datagen = ImageDataGenerator(rescale = 1./255)
test_datagen = ImageDataGenerator(rescale = 1./255)

In [10]:
# Flow training images using train_generator
train_generator = train_datagen.flow_from_directory(train_dir,
target_size = (150,150),
class_mode ="categorical")

Found 1700 images belonging to 3 classes.


In [11]:
# Flow testing images using test_generator
test_generator = test_datagen.flow_from_directory(test_dir,
target_size = (150,150),
class_mode = "categorical")

Found 372 images belonging to 3 classes.


In [12]:
#Building the model
inputs_ = keras.Input(shape=(150,150,3))
conv_1 = keras.layers.Conv2D(32, (2,2), activation = "relu")(inputs_)
pool_1 = keras.layers.MaxPooling2D(2,2)(conv_1)
conv_2 = keras.layers.Conv2D(64, (2,2), activation="relu")(pool_1)
pool_2 = keras.layers.MaxPooling2D(2,2)(conv_2)
conv_3 = keras.layers.Conv2D(128, (2,2), activation="relu")(pool_2)
pool_3 = keras.layers.MaxPooling2D(2,2)(conv_3)
conv_4 = keras.layers.Conv2D(256, (2,2), activation = "relu")(pool_3)
pool_4 =keras.layers.MaxPooling2D(2,2)(conv_4)
#Flatten the results to feed into a DNN
flatten = keras.layers.Flatten()(pool_4)
# 512 neuron hidden layer
dense_1 = keras.layers.Dense(512, activation = "relu")(flatten)
# output layer will have 3 units one for each class
output = keras.layers.Dense(3, activation = "softmax")(dense_1)
model = keras.Model(inputs = inputs_, outputs = output)
#print Model Summary
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 149, 149, 32)   │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 73, 73, 64)     │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 35, 35, 128)    │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 256)    │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     8,389,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,563,555 (32.67 MB)

 Trainable params: 8,563,555 (32.67 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
#Compiling the model
model.compile(optimizer = "adam",
loss = "categorical_crossentropy",
metrics = ["accuracy"])

In [14]:
#Training
history = model.fit(train_generator,
validation_data = test_generator,
epochs = 10,
verbose = 2)

Epoch 1/10
54/54 - 17s - 318ms/step - accuracy: 0.7576 - loss: 0.5393 - val_accuracy: 0.7876 - val_loss: 0.6562
Epoch 2/10
54/54 - 7s - 131ms/step - accuracy: 0.9965 - loss: 0.0162 - val_accuracy: 0.9005 - val_loss: 0.4733
Epoch 3/10
54/54 - 7s - 125ms/step - accuracy: 1.0000 - loss: 0.0013 - val_accuracy: 0.8522 - val_loss: 0.8807
Epoch 4/10
54/54 - 7s - 125ms/step - accuracy: 1.0000 - loss: 1.8125e-04 - val_accuracy: 0.8306 - val_loss: 1.0146
Epoch 5/10
54/54 - 7s - 130ms/step - accuracy: 1.0000 - loss: 5.9508e-05 - val_accuracy: 0.8306 - val_loss: 1.0435
Epoch 6/10
54/54 - 6s - 116ms/step - accuracy: 1.0000 - loss: 4.1722e-05 - val_accuracy: 0.8306 - val_loss: 1.1017
Epoch 7/10
54/54 - 7s - 127ms/step - accuracy: 1.0000 - loss: 2.3864e-05 - val_accuracy: 0.8306 - val_loss: 1.1729
Epoch 8/10
54/54 - 6s - 116ms/step - accuracy: 1.0000 - loss: 1.3500e-05 - val_accuracy: 0.8253 - val_loss: 1.2803
Epoch 9/10
54/54 - 7s - 131ms/step - accuracy: 1.0000 - loss: 7.6647e-06 - val_accuracy: 0.